In [13]:
import spacy
from rank_bm25 import BM25Okapi

In [14]:
# 1. 모델 로드 (불필요한 파서, NER 기능은 제외하여 속도 향상)
nlp = spacy.load("../models/en_core_web_sm", disable=["parser", "ner"])
nlp

In [16]:
# 2. 샘플 데이터 (검색 대상 문서들)
doc1 = """The Light Running Margin (LRM) is a critical design parameter in naval architecture that accounts for the inevitable increase in resistance a vessel faces over its operational lifetime. 
When a ship is newly built, its hull is smooth and its propeller is clean, allowing it to achieve design speeds with minimum effort. 
However, as the ship ages, factors such as hull fouling (the accumulation of marine growth) and surface roughening due to corrosion significantly increase the power required to maintain the same speed. 
The LRM ensures the engine is calibrated to handle these "heavy" conditions without exceeding its thermal or mechanical limits.
"""
doc2 = """Technically, the LRM is defined as the percentage difference between the Light Running Curve (the theoretical power/speed relationship of a clean hull in calm seas) and the Layout Curve of the engine. 
In practice, a margin is "baked into" the propeller design so that the propeller absorbs less power than the engine's nominal capacity at a given RPM during initial sea trials. 
This intentional mismatch allows the engine to stay within its optimal operating envelope even when the hull becomes fouled or the ship encounters adverse weather and heavy sea states.
"""
doc3 = """Without an adequate LRM, a vessel would eventually face a "power-limited" scenario. As resistance increases over time, the propeller requires more torque to maintain the same revolutions. 
If the margin is too small, the engine may hit its torque limit before reaching its maximum rated speed, leading to a phenomenon known as "heavy running." 
This not only reduces the ship's maximum attainable speed but also increases fuel consumption and places excessive thermal stress on engine components like cylinders and turbochargers.
"""
doc4 = """Current industry standards, influenced by major engine manufacturers like MAN Energy Solutions and WinGD, typically recommend a Light Running Margin between 3% and 7%, depending on the vessel type and its intended trade route. 
For example, ships operating in tropical waters where marine growth is rapid, or those frequently crossing the North Atlantic's rough seas, may require a higher margin. 
This buffer ensures that the engine's "Operating Point" remains safely to the left of the engine's shop-trial curve for as long as possible.
"""
doc5 = """In the modern era of maritime engineering, the LRM also plays a vital role in environmental compliance and EEDI (Energy Efficiency Design Index) ratings. 
By optimizing the margin, designers can ensure the engine operates at its Minimum Specific Fuel Oil Consumption (SFOC) point for a larger portion of its lifespan. 
Effectively, a well-calculated LRM is a balance between initial trial performance and long-term operational efficiency, ensuring the vessel remains reliable, economical, and compliant from its first voyage to its last.
"""

docs = [doc1, doc2, doc3, doc4, doc5]
docs

['The Light Running Margin (LRM) is a critical design parameter in naval architecture that accounts for the inevitable increase in resistance a vessel faces over its operational lifetime. \nWhen a ship is newly built, its hull is smooth and its propeller is clean, allowing it to achieve design speeds with minimum effort. \nHowever, as the ship ages, factors such as hull fouling (the accumulation of marine growth) and surface roughening due to corrosion significantly increase the power required to maintain the same speed. \nThe LRM ensures the engine is calibrated to handle these "heavy" conditions without exceeding its thermal or mechanical limits.\n',
 'Technically, the LRM is defined as the percentage difference between the Light Running Curve (the theoretical power/speed relationship of a clean hull in calm seas) and the Layout Curve of the engine. \nIn practice, a margin is "baked into" the propeller design so that the propeller absorbs less power than the engine\'s nominal capacit

In [17]:
# 3. 전처리 함수 (토큰화 + 불용어 제거 + 구두점 제거 + 표제어 추출)
def preprocess(text):
    doc = nlp(text.lower())
    # 불용어(stop words)와 구두점을 제외하고 표제어(lemma)를 추출
    return [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

In [21]:
# 4. 코퍼스 토큰화 및 BM25 인덱스 생성
tokenized_corpus = [preprocess(doc) for doc in docs]
bm25 = BM25Okapi(tokenized_corpus)

In [22]:
# 5. 검색 쿼리 실행
query = "what is the range of running rate margine?"
tokenized_query = preprocess(query)
tokenized_query

['range', 'running', 'rate', 'margine']

In [23]:
# 각 문서에 대한 점수 계산
doc_scores = bm25.get_scores(tokenized_query)
doc_scores

array([0.        , 0.        , 1.14460071, 0.        , 0.        ])

In [24]:
# 가장 유사도가 높은 상위 2개 문서 추출
top_n = bm25.get_top_n(tokenized_query, docs, n=3)
top_n

['Without an adequate LRM, a vessel would eventually face a "power-limited" scenario. As resistance increases over time, the propeller requires more torque to maintain the same revolutions. \nIf the margin is too small, the engine may hit its torque limit before reaching its maximum rated speed, leading to a phenomenon known as "heavy running." \nThis not only reduces the ship\'s maximum attainable speed but also increases fuel consumption and places excessive thermal stress on engine components like cylinders and turbochargers.\n',
 'In the modern era of maritime engineering, the LRM also plays a vital role in environmental compliance and EEDI (Energy Efficiency Design Index) ratings. \nBy optimizing the margin, designers can ensure the engine operates at its Minimum Specific Fuel Oil Consumption (SFOC) point for a larger portion of its lifespan. \nEffectively, a well-calculated LRM is a balance between initial trial performance and long-term operational efficiency, ensuring the vesse